# ARS Session 3 — Adaptive Steering Router

This experiment compares four isolated arms on the same deterministic GSM8K subset: untouched Phi-2, the fixed V2 checkpoint, a validation-selected static action from the fresh steering bank, and the prompt-conditioned router. Phi-2 remains frozen.

The default run collects fresh rollouts and trains a fresh bank and router. Attached rollout files can be reused only by setting an explicit directory and enabling the reuse switch. Existing steering banks are never discovered automatically; reuse requires an explicit checkpoint override. Phase failures are recorded so the notebook reaches its final status cell.


## 1. Execution configuration

No previous steering checkpoint, bank, rollout, or norm-calibration file is required. The notebook builds fresh training artifacts in a unique output directory and packages them into a ZIP for later evaluation.


In [ ]:
# ==========================================
# EXECUTION & SCALE CONFIGURATION
# ==========================================
import os
from pathlib import Path
from datetime import datetime

SMOKE_TEST = False
ATTENTION_IMPLEMENTATION = "sdpa"
V2_ROUTE = [12, 15, 18, 21]
CANDIDATE_LAYERS = [8, 11, 12, 13, 15, 17, 18, 19, 20, 21, 22, 23, 24, 26]
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_ROOT = Path("/kaggle/working/ars_session3")
OUTPUT_DIR = OUTPUT_ROOT / RUN_ID
suffix = 1
while OUTPUT_DIR.exists():
    OUTPUT_DIR = OUTPUT_ROOT / f"{RUN_ID}_{suffix}"
    suffix += 1
V2_CHECKPOINT = OUTPUT_DIR / "session2_steernet_v2.pt"
V2_NORM_CALIBRATION = OUTPUT_DIR / "session2_norms_v2.json"
BOOTSTRAP_V2_CHECKPOINT = V2_CHECKPOINT
BOOTSTRAP_V2_NORM_CALIBRATION = V2_NORM_CALIBRATION
BUNDLE_PATH = OUTPUT_ROOT / f"{OUTPUT_DIR.name}_session3_artifacts.zip"
ROUTER_ALPHA_VALUES = [1.0, 1.5, 2.0]

BANK_CHECKPOINT = OUTPUT_DIR / "session3_steering_bank.pt"

ROUTER_CHECKPOINT = OUTPUT_DIR / "session3_router.pt"
EXPANDED_NORM_CALIBRATION = OUTPUT_DIR / "candidate_norms_p99.json"
STATIC_ACTION_FILE = OUTPUT_DIR / "session3_static_bank_action.json"
DIVERSIFY_PROMPTS = True
PREFERENCE_WEIGHT = 0.25
TASK_VALIDATION_ROWS = 32
OOD_DATASETS = []
OOD_N = 100
if SMOKE_TEST:
    CALIB_N, GATE0_QUESTIONS, COLLECTION_QUESTIONS, COLLECTION_K = 24, 10, 40, 2
    V2_BOOTSTRAP_EPOCHS = 1
    V2_BOOTSTRAP_EVAL_QUESTIONS = 8
    BANK_EPOCHS, MAX_RFT_QUESTIONS = 1, 40
    BANK_ROUTE_EXAMPLES_PER_LAYER, BANK_SCREEN_ROWS = 8, 8
    MAX_PAIR_ROUTES, MAX_TRIPLE_ROUTES, MAX_QUAD_ROUTES = 3, 2, 1
    MAX_ROUTER_ROWS, ROUTER_EPOCHS = 12, 1
    TASK_VALIDATION_ROWS = 8
    OOD_DATASETS, OOD_N = [], 24
    STATIC_SCREEN_QUESTIONS = 4
    ROUTE_COST = 0.01
    BENCH_N_TEST, BENCH_SHOTS = 24, [0]
else:
    CALIB_N, GATE0_QUESTIONS, COLLECTION_QUESTIONS, COLLECTION_K = 128, 60, 2000, 8
    V2_BOOTSTRAP_EPOCHS = 2
    V2_BOOTSTRAP_EVAL_QUESTIONS = 100
    BANK_EPOCHS, MAX_RFT_QUESTIONS = 2, 2000
    BANK_ROUTE_EXAMPLES_PER_LAYER, BANK_SCREEN_ROWS = 256, 32
    MAX_PAIR_ROUTES, MAX_TRIPLE_ROUTES, MAX_QUAD_ROUTES = 12, 8, 6
    MAX_ROUTER_ROWS, ROUTER_EPOCHS = 384, 3
    STATIC_SCREEN_QUESTIONS = 48
    ROUTE_COST = 0.01
    BENCH_N_TEST, BENCH_SHOTS = 200, [0, 2]

print({
    "smoke_test": SMOKE_TEST,
    "fresh_artifacts_only": True,
    "fresh_collection_required": True,
    "fresh_bootstrap_required": True,
    "v2_bootstrap_eval_questions": V2_BOOTSTRAP_EVAL_QUESTIONS,
    "diversify_prompts": DIVERSIFY_PROMPTS,
    "fixed_route": V2_ROUTE,
    "candidate_layers": CANDIDATE_LAYERS,
    "calibration_n": CALIB_N,
    "gate0_n": GATE0_QUESTIONS,
    "bank_epochs": BANK_EPOCHS,
    "route_examples_per_layer": BANK_ROUTE_EXAMPLES_PER_LAYER,
    "route_caps": [MAX_PAIR_ROUTES, MAX_TRIPLE_ROUTES, MAX_QUAD_ROUTES],
    "router_epochs": ROUTER_EPOCHS,
    "router_alpha_values": ROUTER_ALPHA_VALUES,
    "router_teacher_questions": MAX_ROUTER_ROWS,
    "task_validation_rows": TASK_VALIDATION_ROWS,
    "preference_weight": PREFERENCE_WEIGHT,
    "static_screen_questions": STATIC_SCREEN_QUESTIONS,
    "ood_datasets": OOD_DATASETS,
    "ood_questions_each": OOD_N,
    "benchmark_n": BENCH_N_TEST,
    "shots": BENCH_SHOTS,
    "bank_checkpoint": str(BANK_CHECKPOINT),
    "output_dir": str(OUTPUT_DIR), "artifact_bundle": str(BUNDLE_PATH),
}, flush=True)


## 2. Kaggle checkout and live progress

Training uses one visible T4 at a time for predictable memory use. If collection is needed, its two workers each get one GPU. The stream helper forwards tqdm refreshes to the cell immediately.


In [ ]:
import codecs
import subprocess
import sys
import threading

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["ARS_ATTENTION_IMPLEMENTATION"] = ATTENTION_IMPLEMENTATION
REPO_URL = "https://github.com/Abdullah182155/adaptive-residual-steering.git"
REPO_DIR = Path("/kaggle/working/adaptive-residual-steering")
BRANCH = "development"
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

def _stream_child(prefix, command, env=None):
    """Stream progress refreshes and return the child status."""
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env, bufsize=0)
    pending = bytearray()
    while True:
        value = process.stdout.read(1)
        if not value: break
        if value in (b"\r", b"\n"):
            if pending:
                print(prefix + codecs.decode(pending, "utf-8", errors="replace"), flush=True)
                pending.clear()
        else:
            pending.extend(value)
    if pending: print(prefix + codecs.decode(pending, "utf-8", errors="replace"), flush=True)
    return process.wait()

def run_live(command, *, env=None, prefix=""):
    code = _stream_child(prefix, command, env)
    if code: raise RuntimeError(f"{prefix}child exited with code {code}: {command}")

def run_parallel_live(jobs):
    results = {}
    def worker(job_id, prefix, command, env): results[job_id] = _stream_child(prefix, command, env)
    threads = [threading.Thread(target=worker, args=(i, *job), daemon=True) for i, job in enumerate(jobs)]
    for thread in threads: thread.start()
    for thread in threads: thread.join()
    failures = [f"job {i} exited with code {value}" for i, value in results.items() if value]
    if failures: raise RuntimeError("; ".join(failures))

print(f"Repository ready: {REPO_DIR} ({BRANCH})", flush=True)


## 3. Runtime, inputs, and phase tracking

Failures are recorded in the phase status dictionary. Missing checkpoints or data skip dependent phases while allowing the final summary to run.


In [ ]:
import hashlib
import json

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PHASE_STATUS = {name: "pending" for name in (
    "runtime", "tests", "inputs", "rollouts", "v2_bootstrap", "norm_calibration",
    "steering_bank", "router", "benchmark", "artifact_bundle"
)}
ROLLOUT_PROVENANCE = "fresh_collection"
try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, check=True
    )
    visible_gpus = [line.strip() for line in result.stdout.splitlines() if line.strip()]
    PHASE_STATUS["runtime"] = "passed" if visible_gpus else "no_gpu_detected"
    print("Visible GPUs:", visible_gpus, flush=True)
except Exception as exc:
    visible_gpus = []
    PHASE_STATUS["runtime"] = f"failed: {type(exc).__name__}: {exc}"
    print(PHASE_STATUS["runtime"], flush=True)

try:
    result = subprocess.run(
        [sys.executable, "-u", "-m", "pytest", "tests/", "-x", "-q"],
        check=False
    )
    PHASE_STATUS["tests"] = "passed" if result.returncode == 0 else f"failed_exit_{result.returncode}"
except Exception as exc:
    PHASE_STATUS["tests"] = f"failed: {type(exc).__name__}: {exc}"
print("Tests:", PHASE_STATUS["tests"], flush=True)

missing_v2_norm_layers = list(V2_ROUTE)
PHASE_STATUS["inputs"] = "bootstrap_required"
print({
    "checkpoint": str(V2_CHECKPOINT),
    "checkpoint_exists": V2_CHECKPOINT.is_file(),
    "v2_norms": str(V2_NORM_CALIBRATION),
    "v2_norms_exists": V2_NORM_CALIBRATION.is_file(),
    "missing_v2_norm_layers": missing_v2_norm_layers,
    "fresh_training_required": True,
    "external_checkpoints_used": False,
}, flush=True)

def find_complete_rollout_pair(search_roots):
    complete = []
    for root in dict.fromkeys(Path(root) for root in search_roots if root is not None):
        if not root.exists():
            continue
        for extension in ("jsonl", "json"):
            for shard0 in root.glob(f"**/shard0.{extension}"):
                shard1 = shard0.with_name(f"shard1.{extension}")
                if shard1.is_file():
                    complete.append((max(p.stat().st_mtime for p in (shard0, shard1)), [shard0, shard1]))
    return max(complete, key=lambda item: item[0])[1] if complete else []

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

current_run_rollouts = find_complete_rollout_pair([OUTPUT_DIR / "rollouts"])
if current_run_rollouts:
    ROLLOUT_FILES = current_run_rollouts
    ROLLOUT_PROVENANCE = "resumed_current_run"
else:
    ROLLOUT_FILES = []
    ROLLOUT_PROVENANCE = "fresh_collection"

print("Rollout files:", [str(path) for path in ROLLOUT_FILES], flush=True)
print("Rollout provenance:", ROLLOUT_PROVENANCE, flush=True)
PHASE_STATUS["v2_bootstrap"] = "not_needed" if PHASE_STATUS["inputs"] == "ready" else "pending"


## 4. Fresh rollouts by default

This run never loads attached rollout shards, old checkpoints, or a previous steering bank. It may resume only a complete shard pair already inside this run's unique output folder. Otherwise Gate 0 and fresh collection run. Collection uses two GPUs in parallel when available and runs the shards sequentially on one GPU.


In [ ]:
if len(ROLLOUT_FILES) == 2:
    PHASE_STATUS["rollouts"] = "resumed_current_run_shards"
else:
    try:
        rollout_output = OUTPUT_DIR / "rollouts"
        rollout_output.mkdir(parents=True, exist_ok=True)
        gate0_command = [
            sys.executable, "-u", "scripts/run_rft_pipeline.py", "--stage", "gate0",
            "--n_questions", str(GATE0_QUESTIONS), "--k", str(COLLECTION_K),
            "--temperature", "0.7", "--gate0_nonblocking",
            "--output_dir", str(rollout_output),
        ]
        gate0_env = os.environ.copy()
        gate0_env["CUDA_VISIBLE_DEVICES"] = "0"
        print(f"Running diagnostic Gate 0 on {GATE0_QUESTIONS} questions before fresh collection", flush=True)
        run_live(gate0_command, env=gate0_env, prefix="[Gate 0] ")
        gate0_stats = json.loads((rollout_output / "gate0_stats.json").read_text(encoding="utf-8"))
        print({
            key: gate0_stats.get(key)
            for key in ("n", "pass_at_k_lenient", "greedy_lenient_accuracy", "quality_gate_passed")
        }, flush=True)

        jobs = []
        for shard, gpu in ((0, "0"), (1, "1")):
            env = os.environ.copy()
            env["CUDA_VISIBLE_DEVICES"] = gpu
            command = [
                sys.executable, "-u", "scripts/run_rft_pipeline.py", "--stage", "collect",
                "--n_questions", str(COLLECTION_QUESTIONS), "--k", str(COLLECTION_K),
                "--temperature", "0.7", "--n_shards", "2", "--shard", str(shard),
                "--allow_low_yield", "--output_dir", str(rollout_output),
            ]
            jobs.append((f"[GPU{gpu}/shard{shard}] ", command, env))

        if len(visible_gpus) >= 2:
            run_parallel_live(jobs)
        elif len(visible_gpus) == 1:
            print("One GPU detected; collecting rollout shards sequentially.", flush=True)
            for shard in (0, 1):
                env = os.environ.copy()
                env["CUDA_VISIBLE_DEVICES"] = "0"
                command = [
                    sys.executable, "-u", "scripts/run_rft_pipeline.py", "--stage", "collect",
                    "--n_questions", str(COLLECTION_QUESTIONS), "--k", str(COLLECTION_K),
                    "--temperature", "0.7", "--n_shards", "2", "--shard", str(shard),
                    "--allow_low_yield", "--output_dir", str(rollout_output),
                ]
                run_live(command, env=env, prefix=f"[GPU0/shard{shard}] ")
        else:
            raise RuntimeError("No visible GPU is available for rollout collection.")

        ROLLOUT_FILES = find_complete_rollout_pair([rollout_output])
        ROLLOUT_PROVENANCE = "fresh_collection" if len(ROLLOUT_FILES) == 2 else "collection_incomplete"
        PHASE_STATUS["rollouts"] = "collected" if len(ROLLOUT_FILES) == 2 else "collection_outputs_missing"
    except Exception as exc:
        PHASE_STATUS["rollouts"] = f"failed: {type(exc).__name__}: {exc}"
        print("Rollout phase failed; dependent phases will be skipped:", exc, flush=True)
print("Rollout status:", PHASE_STATUS["rollouts"], flush=True)


## 5. Recalibrate missing V2 norms or rebuild missing V2 artifacts

If the V2 checkpoint exists but its p99 calibration is incomplete for layers 12, 15, 18, and 21, the notebook recalibrates those norms from the selected rollout questions and keeps the existing checkpoint. It trains a replacement V2 checkpoint only when the checkpoint itself is missing.


In [ ]:
if PHASE_STATUS["inputs"] == "bootstrap_required":
    if PHASE_STATUS["rollouts"] in {"resumed_current_run_shards", "collected"} and len(ROLLOUT_FILES) == 2:
        try:
            v2_calibration_command = [sys.executable, "-u", "scripts/calibrate_norms.py",
                                      "--n_samples", str(CALIB_N), "--seed", "42",
                                      "--rollout_files", *map(str, ROLLOUT_FILES),
                                      "--candidate_layers", *map(str, V2_ROUTE),
                                      "--max_length", "768", "--output", str(BOOTSTRAP_V2_NORM_CALIBRATION),
                                      "--device", "cuda"]
            env = os.environ.copy()
            env["CUDA_VISIBLE_DEVICES"] = "0"
            run_live(v2_calibration_command, env=env, prefix="[V2 norms] ")
            if not BOOTSTRAP_V2_NORM_CALIBRATION.is_file():
                raise FileNotFoundError(f"V2 norm calibration was not produced: {BOOTSTRAP_V2_NORM_CALIBRATION}")
            V2_NORM_CALIBRATION = BOOTSTRAP_V2_NORM_CALIBRATION
            v2_train_output = OUTPUT_DIR / "v2_bootstrap"
            v2_train_output.mkdir(parents=True, exist_ok=True)
            v2_train_command = [sys.executable, "-u", "scripts/run_rft_pipeline.py",
                                "--stage", "train", "--skip_collection", "--steernet_arch", "v2",
                                "--norm_calibration", str(BOOTSTRAP_V2_NORM_CALIBRATION),
                                "--rollout_dir", str(ROLLOUT_FILES[0].parent),
                                "--epochs", str(V2_BOOTSTRAP_EPOCHS),
                                "--eval_questions", str(V2_BOOTSTRAP_EVAL_QUESTIONS),
                                "--checkpoint", str(BOOTSTRAP_V2_CHECKPOINT),
                                "--output_dir", str(v2_train_output)]
            if DIVERSIFY_PROMPTS:
                v2_train_command.append("--diversify_prompt_templates")
            run_live(v2_train_command, env=env, prefix="[V2 bootstrap] ")
            if not BOOTSTRAP_V2_CHECKPOINT.is_file():
                raise FileNotFoundError(f"Fresh V2 bootstrap did not produce a checkpoint: {BOOTSTRAP_V2_CHECKPOINT}")
            V2_CHECKPOINT = BOOTSTRAP_V2_CHECKPOINT
            PHASE_STATUS["v2_bootstrap"] = "trained_from_fresh_rollouts"
            PHASE_STATUS["inputs"] = "bootstrapped"
        except Exception as exc:
            PHASE_STATUS["v2_bootstrap"] = f"failed: {type(exc).__name__}: {exc}"
            PHASE_STATUS["inputs"] = "bootstrap_failed"
            print("V2 bootstrap failed; dependent training will be skipped, but the notebook will continue:", exc, flush=True)
    else:
        PHASE_STATUS["v2_bootstrap"] = "skipped_no_rollouts"
        PHASE_STATUS["inputs"] = "bootstrap_failed"
elif PHASE_STATUS["inputs"] == "bootstrapped":
    PHASE_STATUS["v2_bootstrap"] = "trained_from_fresh_rollouts"
print("V2 bootstrap status:", PHASE_STATUS["v2_bootstrap"], flush=True)


## 6. Calibrate candidate p99 norms

This uses rollout questions only. The original V2 norm file remains separate so the fixed V2 arm retains its calibrated caps.


In [ ]:
if PHASE_STATUS["rollouts"] in {"resumed_current_run_shards", "collected"} and PHASE_STATUS["inputs"] in {"ready", "bootstrapped"}:
    try:
        command = [sys.executable, "-u", "scripts/calibrate_norms.py", "--n_samples", str(CALIB_N),
                   "--seed", "42", "--rollout_files", *[str(path) for path in ROLLOUT_FILES],
                   "--candidate_layers", *[str(layer) for layer in CANDIDATE_LAYERS],
                   "--max_length", "768", "--output", str(EXPANDED_NORM_CALIBRATION), "--device", "cuda"]
        env = os.environ.copy(); env["CUDA_VISIBLE_DEVICES"] = "0"
        run_live(command, env=env, prefix="[norms] ")
        PHASE_STATUS["norm_calibration"] = "passed" if EXPANDED_NORM_CALIBRATION.is_file() else "output_missing"
    except Exception as exc:
        PHASE_STATUS["norm_calibration"] = f"failed: {type(exc).__name__}: {exc}"
        print(PHASE_STATUS["norm_calibration"], flush=True)
else:
    PHASE_STATUS["norm_calibration"] = "skipped_missing_prerequisites"


## 7. Train the steering bank by route curriculum

The bank is trained in stages: each candidate layer alone, then screened pairs, triples, and quads. The fixed V2 route is preserved in the router action catalog as an anchor. The current bank still shares one adapter per layer across routes; the report will treat route-specific cooperation as unproven. Phi-2 remains frozen.


In [ ]:
if PHASE_STATUS["norm_calibration"] == "passed" and PHASE_STATUS["inputs"] in {"ready", "bootstrapped"} and ROLLOUT_FILES:
    try:
        command = [
            sys.executable, "-u", "scripts/run_session3_router.py", "--stage", "train_bank",
            "--output_dir", str(OUTPUT_DIR), "--candidate_layers", *map(str, CANDIDATE_LAYERS),
            "--norm_calibration", str(EXPANDED_NORM_CALIBRATION),
            "--v2_norm_calibration", str(V2_NORM_CALIBRATION), "--v2_checkpoint", str(V2_CHECKPOINT),
            "--bank_checkpoint", str(BANK_CHECKPOINT), "--rollout_files", *map(str, ROLLOUT_FILES),
            "--epochs", str(BANK_EPOCHS), "--max_questions", str(MAX_RFT_QUESTIONS),
            "--route_examples_per_layer", str(BANK_ROUTE_EXAMPLES_PER_LAYER),
            "--screen_validation_rows", str(BANK_SCREEN_ROWS),
            "--max_pair_routes", str(MAX_PAIR_ROUTES),
            "--max_triple_routes", str(MAX_TRIPLE_ROUTES),
            "--max_quad_routes", str(MAX_QUAD_ROUTES),
            "--preference_weight", str(PREFERENCE_WEIGHT),
        ]
        if DIVERSIFY_PROMPTS:
            command.append("--diversify_prompts")
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = "0"
        run_live(command, env=env, prefix="[bank] ")
        PHASE_STATUS["steering_bank"] = "passed" if BANK_CHECKPOINT.is_file() else "checkpoint_missing"
    except Exception as exc:
        PHASE_STATUS["steering_bank"] = f"failed: {type(exc).__name__}: {exc}"
        print(PHASE_STATUS["steering_bank"], flush=True)
else:
    PHASE_STATUS["steering_bank"] = "skipped_prerequisite_failure"


## 8. Train the prompt-conditioned router

The router sees prompt tokens only. For each identical question prompt, the teacher scores the full route and alpha catalog, including k=0, on every available verified completion and averages those scores before creating one target. Completions for the same question share a requested shot count and template; if a long completion forces fewer shots, its exact tokenized prompt is grouped separately. The target report includes completion counts, action-score spread, ties, and shot/template metadata. Train/validation splits remain question-disjoint. Router checkpoint selection uses the same prompt shield and route-cost objective as teacher scoring, and reports validation action diversity and quality separately for each effective shot count.


In [ ]:
if PHASE_STATUS["steering_bank"] == "passed" and ROLLOUT_FILES:
    try:
        command = [
            sys.executable, "-u", "scripts/run_session3_router.py", "--stage", "train_router",
            "--output_dir", str(OUTPUT_DIR), "--candidate_layers", *map(str, CANDIDATE_LAYERS),
            "--norm_calibration", str(EXPANDED_NORM_CALIBRATION),
            "--v2_norm_calibration", str(V2_NORM_CALIBRATION), "--v2_checkpoint", str(V2_CHECKPOINT),
            "--bank_checkpoint", str(BANK_CHECKPOINT), "--router_checkpoint", str(ROUTER_CHECKPOINT),
            "--rollout_files", *map(str, ROLLOUT_FILES), "--epochs", str(ROUTER_EPOCHS),
            "--max_questions", str(MAX_RFT_QUESTIONS), "--max_router_rows", str(MAX_ROUTER_ROWS),
            "--route_cost", str(ROUTE_COST), "--router_alphas", *map(str, ROUTER_ALPHA_VALUES),
            "--static_screen_questions", str(STATIC_SCREEN_QUESTIONS),
            "--task_validation_rows", str(TASK_VALIDATION_ROWS),
        ]
        if DIVERSIFY_PROMPTS:
            command.append("--diversify_prompts")
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = "0"
        run_live(command, env=env, prefix="[router] ")
        PHASE_STATUS["router"] = "passed" if ROUTER_CHECKPOINT.is_file() else "checkpoint_missing"
    except Exception as exc:
        PHASE_STATUS["router"] = f"failed: {type(exc).__name__}: {exc}"
        print(PHASE_STATUS["router"], flush=True)
else:
    PHASE_STATUS["router"] = "skipped_prerequisite_failure"


## 9. Compare the four arms on one paired subset

All arms share the same seeded questions, shot conditions, prompt-template schedule, greedy decoding, and repetition penalty. The static bank route and alpha are selected on held-out rollout validation rows, never on benchmark questions. The output directory contains arm-level JSONL predictions and paired summaries.


In [ ]:
BENCH_RECORD = OUTPUT_DIR / f"session3_gsm8k_{BENCH_N_TEST}_seed42.json"
if PHASE_STATUS["inputs"] in {"ready", "bootstrapped"}:
    try:
        command = [
            sys.executable, "-u", "scripts/run_session3_router.py", "--stage", "benchmark",
            "--output_dir", str(OUTPUT_DIR), "--candidate_layers", *map(str, CANDIDATE_LAYERS),
            "--norm_calibration", str(EXPANDED_NORM_CALIBRATION),
            "--v2_norm_calibration", str(V2_NORM_CALIBRATION), "--v2_checkpoint", str(V2_CHECKPOINT),
            "--bank_checkpoint", str(BANK_CHECKPOINT), "--router_checkpoint", str(ROUTER_CHECKPOINT),
            "--bench_n", str(BENCH_N_TEST), "--bench_seed", "42", "--shots", *map(str, BENCH_SHOTS),
            "--require_all_arms", "--ood_n", str(OOD_N),
        ]
        if OOD_DATASETS:
            command.extend(["--ood_datasets", *OOD_DATASETS])
        if DIVERSIFY_PROMPTS:
            command.append("--diversify_prompts")
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = "0"
        run_live(command, env=env, prefix="[benchmark] ")
        if BENCH_RECORD.is_file():
            report = json.loads(BENCH_RECORD.read_text(encoding="utf-8"))
            expected_arms = ["baseline", "steer_only", "static_bank", "router"]
            shots_complete = all(
                set(expected_arms).issubset(report.get("shots", {}).get(str(shot), {}))
                and all(
                    report["shots"][str(shot)][arm].get("n") == BENCH_N_TEST
                    for arm in expected_arms
                )
                for shot in BENCH_SHOTS
            )
            four_arms_complete = (
                report.get("four_arm_complete") is True
                and report.get("evaluated_arms") == expected_arms
                and shots_complete
            )
            suite_status_path = OUTPUT_DIR / "session3_benchmark_status.json"
            suite_status = json.loads(suite_status_path.read_text(encoding="utf-8")) if suite_status_path.is_file() else {}
            suites = suite_status.get("suites", {})
            ood_complete = all(
                suites.get(name, {}).get("status") == "passed_four_arms"
                for name in OOD_DATASETS
            )
            if four_arms_complete and ood_complete:
                PHASE_STATUS["benchmark"] = "passed_four_arms_and_ood"
            elif four_arms_complete:
                PHASE_STATUS["benchmark"] = "passed_four_arms_ood_incomplete"
            else:
                PHASE_STATUS["benchmark"] = "failed_incomplete_four_arm_record"
            print({"evaluated_arms": report.get("evaluated_arms"), "four_arms_complete": four_arms_complete,
                   "ood_status": suites}, flush=True)
        else:
            PHASE_STATUS["benchmark"] = "report_missing"
    except Exception as exc:
        PHASE_STATUS["benchmark"] = f"failed: {type(exc).__name__}: {exc}"
        print(PHASE_STATUS["benchmark"], flush=True)
else:
    PHASE_STATUS["benchmark"] = "skipped_missing_v2_artifacts"
print("Benchmark status:", PHASE_STATUS["benchmark"], flush=True)


## 10. Final status and artifacts

A completed notebook does not imply the router improved accuracy. Keep it only if the paired results show useful improvement over both baseline and fixed V2.


In [ ]:
from src.training.adaptive_data import TASK_SELECTION_SHOT_WEIGHTS

summary = {
    "run_id": RUN_ID,
    "run_dir": str(OUTPUT_DIR),
    "git_branch": BRANCH,
    "git_commit": subprocess.run(
        ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
        capture_output=True, text=True, check=True,
    ).stdout.strip(),
    "run_config": {
        "smoke_test": SMOKE_TEST,
        "attention_implementation": ATTENTION_IMPLEMENTATION,
        "collection_questions": COLLECTION_QUESTIONS,
        "collection_k": COLLECTION_K,
        "bank_epochs": BANK_EPOCHS,
        "router_epochs": ROUTER_EPOCHS,
      "router_alpha_values": ROUTER_ALPHA_VALUES,
        "router_teacher_questions": MAX_ROUTER_ROWS,
        "task_validation_rows": TASK_VALIDATION_ROWS,
        "preference_weight": PREFERENCE_WEIGHT,
        "task_selection_shot_weights": {str(shot): weight for shot, weight in TASK_SELECTION_SHOT_WEIGHTS.items()},
        "task_selection_active_weights": {
            str(shot): TASK_SELECTION_SHOT_WEIGHTS[shot] / sum(TASK_SELECTION_SHOT_WEIGHTS[value] for value in BENCH_SHOTS)
            for shot in BENCH_SHOTS
        },
        "route_cost": ROUTE_COST,
        "static_screen_questions": STATIC_SCREEN_QUESTIONS,
        "benchmark_questions": BENCH_N_TEST,
        "shots": BENCH_SHOTS,
        "ood_datasets": OOD_DATASETS,
        "ood_questions_each": OOD_N,
        "fresh_training_only": True,
        "seed": 42,
    },
    "phase_status": PHASE_STATUS,
    "artifact_bundle_path": str(BUNDLE_PATH),
    "rollout_provenance": ROLLOUT_PROVENANCE,
    "v2_checkpoint": str(V2_CHECKPOINT),
    "v2_checkpoint_sha256": sha256_file(V2_CHECKPOINT) if V2_CHECKPOINT.is_file() else None,
    "v2_norm_calibration": str(V2_NORM_CALIBRATION),
    "v2_norm_sha256": sha256_file(V2_NORM_CALIBRATION) if V2_NORM_CALIBRATION.is_file() else None,
    "expanded_norm_calibration": str(EXPANDED_NORM_CALIBRATION) if EXPANDED_NORM_CALIBRATION.is_file() else None,
    "expanded_norm_sha256": sha256_file(EXPANDED_NORM_CALIBRATION) if EXPANDED_NORM_CALIBRATION.is_file() else None,
    "v2_artifact_provenance": PHASE_STATUS.get("v2_bootstrap"),
    "fixed_route": V2_ROUTE,
    "router_candidates": CANDIDATE_LAYERS,
    "rollout_files": [
        {"path": str(path), "sha256": sha256_file(path)}
        for path in ROLLOUT_FILES if Path(path).is_file()
    ],
    "bank_checkpoint": str(BANK_CHECKPOINT) if BANK_CHECKPOINT.is_file() else None,
    "bank_sha256": sha256_file(BANK_CHECKPOINT) if BANK_CHECKPOINT.is_file() else None,
    "static_bank_action": str(STATIC_ACTION_FILE) if STATIC_ACTION_FILE.is_file() else None,
    "static_bank_action_sha256": sha256_file(STATIC_ACTION_FILE) if STATIC_ACTION_FILE.is_file() else None,
    "router_checkpoint": str(ROUTER_CHECKPOINT) if ROUTER_CHECKPOINT.is_file() else None,
    "router_sha256": sha256_file(ROUTER_CHECKPOINT) if ROUTER_CHECKPOINT.is_file() else None,
    "benchmark_reports": [str(path) for path in sorted(OUTPUT_DIR.glob("session3_*_seed42.json"))],
    "benchmark_sha256": sha256_file(BENCH_RECORD) if BENCH_RECORD.is_file() else None,
    "benchmark_sha256s": {path.name: sha256_file(path) for path in sorted(OUTPUT_DIR.glob("session3_*_seed42.json"))},
    "benchmark_suite_status": str(OUTPUT_DIR / "session3_benchmark_status.json") if (OUTPUT_DIR / "session3_benchmark_status.json").is_file() else None,
    "per_question_files": [
        str(path) for path in (OUTPUT_DIR / "generations").glob("*.jsonl")
    ] if (OUTPUT_DIR / "generations").exists() else [],
}
(OUTPUT_DIR / "session3_status.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2), flush=True)


## 11. One-cell results, statistics, and diagnosis

Run this after the session finishes. It prints the baseline, fixed V2, validation-selected static bank action, and router results, with paired comparisons, confidence intervals, route usage, and artifact hashes. Copy the JSON output and send it as one message; individual reports are optional.


In [ ]:
import json
import math
import random
from pathlib import Path

import torch
from src.training.adaptive_data import TASK_SELECTION_SHOT_WEIGHTS


def _read_json(path):
    path = Path(path)
    if not path.is_file():
        return None
    return json.loads(path.read_text(encoding="utf-8"))


def _read_jsonl(path):
    path = Path(path)
    if not path.is_file():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def _mcnemar_p(wins, losses):
    discordant = wins + losses
    if discordant == 0:
        return 1.0
    tail = sum(math.comb(discordant, index) for index in range(min(wins, losses) + 1))
    return min(1.0, 2.0 * tail / (2 ** discordant))


def _paired_diagnostics(control, treatment, seed=42, bootstrap_samples=2000):
    common = sorted(set(control) & set(treatment))
    if not common:
        return {"n": 0, "status": "no matched question IDs"}
    differences = [int(bool(treatment[key])) - int(bool(control[key])) for key in common]
    wins = sum(value == 1 for value in differences)
    losses = sum(value == -1 for value in differences)
    ties = len(differences) - wins - losses
    rng = random.Random(seed)
    draws = sorted(
        100.0 * sum(rng.choice(differences) for _ in differences) / len(differences)
        for _ in range(bootstrap_samples)
    )
    return {
        "n": len(common),
        "wins": wins,
        "losses": losses,
        "ties": ties,
        "delta_pp": round(100.0 * sum(differences) / len(differences), 3),
        "bootstrap_ci95_pp": [round(draws[int(0.025 * bootstrap_samples)], 3),
                               round(draws[min(bootstrap_samples - 1, int(0.975 * bootstrap_samples))], 3)],
        "mcnemar_p_value": round(_mcnemar_p(wins, losses), 6),
    }


def _arm_index(filename):
    return {int(row.get("source_index", row.get("idx", i))): bool(row.get("lenient", False))
            for i, row in enumerate(_read_jsonl(OUTPUT_DIR / "generations" / filename))}


def _arm_summary(value):
    if not isinstance(value, dict):
        return None
    return {
        "n": value.get("n"),
        "lenient_accuracy_pct": value.get("lenient_accuracy"),
        "strict_accuracy_pct": value.get("strict_accuracy"),
        "marker_accuracy_pct": value.get("marker_accuracy"),
        "format_rate": value.get("format_rate"),
        "guard_trip_rate": value.get("guard_trip_rate"),
        "nonfinite_count": value.get("nonfinite_count"),
    }


def _route_usage_summary(usage):
    if not isinstance(usage, dict):
        return None
    route_counts = usage.get("route_counts", {})
    total = sum(route_counts.values())
    probabilities = [count / total for count in route_counts.values() if total]
    return {
        "questions": total,
        "unique_routes": len(route_counts),
        "top_routes": dict(sorted(route_counts.items(), key=lambda item: item[1], reverse=True)[:10]),
        "k_counts": usage.get("k_counts", {}),
        "layer_counts": usage.get("layer_counts", {}),
        "alpha_counts": usage.get("alpha_counts", {}),
        "route_entropy": round(-sum(p * math.log(p) for p in probabilities), 4) if probabilities else None,
        "most_used_route_share": round(max(probabilities), 4) if probabilities else None,
    }


active_selection_weights = {int(shot): TASK_SELECTION_SHOT_WEIGHTS[int(shot)] for shot in BENCH_SHOTS}
active_selection_weight_sum = sum(active_selection_weights.values())
report = {
    "run_dir": str(OUTPUT_DIR),
    "selection_objective": {
        "configured_shot_weights": {str(shot): weight for shot, weight in TASK_SELECTION_SHOT_WEIGHTS.items()},
        "evaluated_shots": BENCH_SHOTS,
        "normalized_active_weights": {str(shot): weight / active_selection_weight_sum for shot, weight in active_selection_weights.items()},
    },
    "phase_status": PHASE_STATUS,
    "rollout_provenance": ROLLOUT_PROVENANCE,
    "rollouts": [],
    "training": {},
    "benchmark": {},
    "paired_arm_comparisons": {},
    "diagnosis": [],
}

run_status = _read_json(OUTPUT_DIR / "session3_status.json") or {}
report["provenance"] = {
    key: run_status.get(key) for key in (
        "git_branch", "git_commit", "run_config", "v2_checkpoint_sha256",
        "v2_norm_sha256", "expanded_norm_sha256", "rollout_files",
        "bank_sha256", "static_bank_action_sha256", "router_sha256",
        "benchmark_sha256",
    ) if key in run_status
}

for rollout_path in ROLLOUT_FILES:
    stats_path = Path(rollout_path).with_name(Path(rollout_path).stem + "_stats.json")
    stats = _read_json(stats_path)
    report["rollouts"].append({
        "file": Path(rollout_path).name,
        "n_questions": stats.get("n") if stats else None,
        "k": stats.get("k") if stats else None,
        "pass_at_k_lenient": stats.get("pass_at_k_lenient") if stats else None,
        "verified_positive_rate": stats.get("positive_sample_rate") if stats else None,
        "stats_available": stats is not None,
    })

bank_state = None
if BANK_CHECKPOINT.is_file():
    bank_state = torch.load(BANK_CHECKPOINT, map_location="cpu", weights_only=True)
    curriculum = bank_state.get("curriculum", {})
    stages = curriculum.get("stages", {})
    report["training"]["bank"] = {
        "candidate_layers": bank_state.get("candidate_layers"),
        "depth_bands": curriculum.get("depth_bands"),
        "k0_validation_nll": curriculum.get("k0_validation_nll"),
        "preference_training": curriculum.get("preference_training"),
        "training_orders": curriculum.get("training_orders"),
        "final_catalog_validation_token_mean_nll": curriculum.get("final_catalog_validation_token_mean_nll"),
        "stages": {},
    }
    for stage_name, stage in stages.items():
        scores = stage.get("validation_token_mean_nll", {})
        selected = stage.get("selected", [])
        ranked = sorted(
            ((route, score) for route, score in scores.items() if route and route != "k0"),
            key=lambda item: item[1],
        )
        report["training"]["bank"]["stages"][stage_name] = {
            "candidate_count": len(stage.get("candidates", [])),
            "selected_count": len(selected),
            "selected_routes": selected,
            "screen_questions": stage.get("screen_questions"),
            "screen_rows": stage.get("screen_rows"),
            "selection_metric": stage.get("selection_metric"),
            "validation_pairwise": stage.get("validation_pairwise"),
            "best_scored_routes_by_positive_nll": [
                {"route": route, "validation_token_mean_nll": round(score, 6)}
                for route, score in ranked[:5]
            ],
            "best_pairwise_preference_routes": [
                {"route": route, **metrics}
                for route, metrics in sorted(
                    stage.get("validation_pairwise", {}).get("per_route", {}).items(),
                    key=lambda item: item[1].get("preference_loss", float("inf")),
                )[:5]
            ],
        }
    report["training"]["bank"]["router_catalog_count"] = len(curriculum.get("catalog", []))
else:
    report["training"]["bank"] = {"status": "checkpoint missing"}

report["training"]["static_bank_action"] = _read_json(STATIC_ACTION_FILE)
router_training = _read_json(OUTPUT_DIR / "session3_router_training.json")
if router_training:
    report["training"]["router_teacher"] = {
        "teacher_rows": router_training.get("teacher_labels", {}).get("rows"),
        "teacher_k_counts": router_training.get("teacher_labels", {}).get("k_counts"),
        "teacher_route_counts": router_training.get("teacher_labels", {}).get("route_counts"),
        "teacher_layer_counts": router_training.get("teacher_labels", {}).get("layer_counts"),
        "teacher_alpha_counts": router_training.get("teacher_labels", {}).get("alpha_counts"),
        "route_cost": router_training.get("route_cost"),
        "alpha_values": router_training.get("alpha_values"),
        "best_validation_nll": router_training.get("best_validation_nll"),
        "best_validation_objective": router_training.get("best_validation_objective"),
        "best_checkpoint_selection_metric": router_training.get("best_checkpoint_selection_metric"),
        "best_checkpoint_selection_value": router_training.get("best_checkpoint_selection_value"),
        "teacher_unique_questions": router_training.get("teacher_unique_questions"),
        "teacher_completion_rows_scored": router_training.get("teacher_data", {}).get("completion_rows_scored"),
        "teacher_prompt_groups_with_multiple_completions": router_training.get("teacher_data", {}).get("prompt_groups_with_multiple_completions"),
        "teacher_data_diagnostics": router_training.get("teacher_data"),
        "action_count": router_training.get("action_count"),
        "teacher_target_file": router_training.get("teacher_target_file"),
        "epoch_history": router_training.get("history"),
    }
else:
    report["training"]["router_teacher"] = {"status": "router training report missing"}

benchmark_files = sorted(OUTPUT_DIR.glob("session3_gsm8k_*.json"))
if benchmark_files:
    benchmark_path = benchmark_files[-1]
    benchmark = _read_json(benchmark_path)
    report["benchmark"]["file"] = benchmark_path.name
    report["benchmark"]["test_seed"] = benchmark.get("test_seed")
    report["benchmark"]["evaluated_arms"] = benchmark.get("evaluated_arms")
    report["benchmark"]["expected_arms"] = benchmark.get("expected_arms")
    report["benchmark"]["four_arm_complete"] = benchmark.get("four_arm_complete")
    report["benchmark"]["source_questions"] = len(benchmark.get("test_source_indices", []))
    for shot, values in benchmark.get("shots", {}).items():
        report["benchmark"][shot + "_shot"] = {
            arm: _arm_summary(values.get(arm)) for arm in ("baseline", "steer_only", "static_bank", "router")
            if values.get(arm) is not None
        }
        report["benchmark"][shot + "_shot"]["router_route_usage"] = _route_usage_summary(
            values.get("router_route_usage")
        )
        for paired_key, label in (("paired_steer_only", "steer_only_vs_baseline"),
                                  ("paired_static_bank", "static_bank_vs_baseline"),
                                  ("paired_router", "router_vs_baseline"),
                                  ("static_bank_vs_fixed_v2", "static_bank_vs_fixed_v2"),
                                  ("router_vs_fixed_v2", "router_vs_fixed_v2"),
                                  ("router_vs_static_bank", "router_vs_static_bank")):
            if values.get(paired_key):
                paired = values[paired_key]
                report["benchmark"][shot + "_shot"][label] = {
                    key: paired.get(key) for key in (
                        "fixed_wins", "fixed_losses", "ties", "delta_pp", "ci95_pp",
                        "mcnemar_p_value", "identical_text_rate", "null_intervention_warning",
                    ) if key in paired
                }

        baseline = _arm_index(f"gsm8k_{shot}shot_baseline.jsonl")
        fixed = _arm_index(f"gsm8k_{shot}shot_steer_only.jsonl")
        static = _arm_index(f"gsm8k_{shot}shot_static_bank.jsonl")
        routed = _arm_index(f"gsm8k_{shot}shot_router.jsonl")
        report["paired_arm_comparisons"][shot + "_shot"] = {
            "fixed_v2_vs_baseline": _paired_diagnostics(baseline, fixed),
            "static_bank_vs_baseline": _paired_diagnostics(baseline, static),
            "router_vs_baseline": _paired_diagnostics(baseline, routed),
            "static_bank_vs_fixed_v2": _paired_diagnostics(fixed, static),
            "router_vs_fixed_v2": _paired_diagnostics(fixed, routed),
            "router_vs_static_bank": _paired_diagnostics(static, routed),
        }
else:
    report["benchmark"] = {"status": "benchmark report missing"}

suite_status = _read_json(OUTPUT_DIR / "session3_benchmark_status.json") or {}
report["benchmark"]["suite_status"] = suite_status.get("suites", {})
report["benchmark"]["ood"] = {}
for suite in OOD_DATASETS:
    suite_path = OUTPUT_DIR / f"session3_{suite}_{OOD_N}_seed42.json"
    suite_record = _read_json(suite_path)
    if suite_record:
        report["benchmark"]["ood"][suite] = {
            "file": suite_path.name,
            "source_questions": len(suite_record.get("test_source_indices", [])),
            "four_arm_complete": suite_record.get("four_arm_complete"),
            "shots": {
                shot: {
                    "arms": {
                        arm: _arm_summary(metrics)
                        for arm, metrics in values.items()
                        if arm in ("baseline", "steer_only", "static_bank", "router")
                    },
                    "router_route_usage": _route_usage_summary(values.get("router_route_usage")),
                    "paired": {
                        key: values.get(key)
                        for key in (
                            "paired_steer_only", "paired_static_bank", "paired_router",
                            "static_bank_vs_fixed_v2", "router_vs_fixed_v2",
                            "router_vs_static_bank",
                        )
                        if values.get(key) is not None
                    },
                }
                for shot, values in suite_record.get("shots", {}).items()
            },
        }
    else:
        report["benchmark"]["ood"][suite] = {"status": suite_status.get("suites", {}).get(suite, {}).get("status", "missing")}

for phase, status in PHASE_STATUS.items():
    if any(token in str(status).lower() for token in ("failed", "skipped", "missing", "pending")):
        report["diagnosis"].append(f"{phase}: {status}")

if report.get("benchmark", {}).get("four_arm_complete") is not True:
    report["diagnosis"].append("The GSM8K record does not prove that all four required arms completed with the configured question count.")
for suite, status in report.get("benchmark", {}).get("suite_status", {}).items():
    if status.get("status") != "passed_four_arms":
        report["diagnosis"].append(f"Benchmark suite {suite}: {status.get('status')}")

for shot, comparisons in report["paired_arm_comparisons"].items():
    for comparison_key, label in (
        ("router_vs_static_bank", "router vs static bank"),
        ("router_vs_fixed_v2", "router vs fixed V2"),
        ("router_vs_baseline", "router vs baseline"),
    ):
        comparison = comparisons.get(comparison_key, {})
        delta = comparison.get("delta_pp")
        if delta is None:
            continue
        interval = comparison.get("bootstrap_ci95_pp")
        p_value = comparison.get("mcnemar_p_value")
        report["diagnosis"].append(
            f"{shot}: {label} paired delta={delta:+.2f} pp, "
            f"95% bootstrap CI={interval}, McNemar p={p_value}."
        )
        if comparison_key == "router_vs_static_bank" and shot == "2_shot":
            if delta < 0:
                report["diagnosis"].append(
                    "The router did not beat the static bank in the target 2-shot comparison."
                )
            elif interval and interval[0] <= 0 <= interval[1]:
                report["diagnosis"].append(
                    "The router's 2-shot point estimate is higher than the static bank, "
                    "but this 200-question pilot does not resolve the difference from zero."
                )

usage_entries = [
    values.get("router_route_usage") for key, values in report["benchmark"].items()
    if key.endswith("_shot") and isinstance(values, dict)
]
if usage_entries and not any(item and item.get("questions", 0) for item in usage_entries):
    report["diagnosis"].append("Router benchmark route-usage metadata is empty; check router prefill and arm selection.")
elif usage_entries:
    for shot_name, shot_report in report["benchmark"].items():
        if not shot_name.endswith("_shot") or not isinstance(shot_report, dict):
            continue
        usage = shot_report.get("router_route_usage") or {}
        question_count = int(usage.get("questions") or 0)
        route_share = usage.get("most_used_route_share")
        if question_count and (
            int(usage.get("unique_routes") or 0) <= 1
            or (route_share is not None and float(route_share) >= 0.90)
        ):
            report["diagnosis"].append(
                f"Router route-collapse warning at {shot_name}: "
                f"unique_routes={usage.get('unique_routes')}, "
                f"most_used_route_share={route_share}. Treat this as low diversity, "
                "even if accuracy is competitive."
            )
        k_counts = usage.get("k_counts") or {}
        if question_count and k_counts and max(k_counts.values()) / question_count >= 0.95:
            report["diagnosis"].append(
                f"Router active-k collapse warning at {shot_name}: k_counts={k_counts}."
            )
    report["diagnosis"].append(
        "Router diversity is reported as observed routes, k counts, layer counts, route entropy, "
        "and the most-used-route share; interpret diversity together with paired accuracy."
    )
if not report["diagnosis"]:
    report["diagnosis"].append("All recorded phases completed; compare paired deltas and confidence intervals before selecting a model.")

DIAGNOSTICS_PATH = OUTPUT_DIR / "session3_diagnostics_for_review.json"
required_bundle_files = [
    "session3_status.json",
    "session2_steernet_v2.pt",
    "session2_norms_v2.json",
    "candidate_norms_p99.json",
    "session3_steering_bank.pt",
    "session3_static_bank_action.json",
    "session3_router.pt",
    f"session3_gsm8k_{BENCH_N_TEST}_seed42.json",
    "session3_benchmark_status.json",
    "session3_diagnostics_for_review.json",
    "rollouts/shard0.jsonl",
    "rollouts/shard1.jsonl",
]
missing_bundle_files = [name for name in required_bundle_files if not (OUTPUT_DIR / name).is_file()]
PHASE_STATUS["artifact_bundle"] = (
    "created_complete" if not missing_bundle_files
    else "created_partial_missing: " + ", ".join(missing_bundle_files)
)
report["artifact_bundle"] = {
    "path": str(BUNDLE_PATH),
    "required_files": required_bundle_files,
    "missing_before_packaging": missing_bundle_files,
}
summary = _read_json(OUTPUT_DIR / "session3_status.json") or {}
summary["phase_status"] = PHASE_STATUS
summary["artifact_bundle_path"] = str(BUNDLE_PATH)
summary["artifact_bundle_required_files"] = required_bundle_files
summary["artifact_bundle_missing_before_packaging"] = missing_bundle_files
(OUTPUT_DIR / "session3_status.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
DIAGNOSTICS_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
try:
    from src.utils.artifact_bundle import create_session3_bundle
    BUNDLE_RESULT = create_session3_bundle(
        OUTPUT_DIR, BUNDLE_PATH, required_files=required_bundle_files
    )
    print("Artifact bundle:", json.dumps(BUNDLE_RESULT, ensure_ascii=False), flush=True)
except Exception as exc:
    PHASE_STATUS["artifact_bundle"] = f"failed: {type(exc).__name__}: {exc}"
    summary["phase_status"] = PHASE_STATUS
    (OUTPUT_DIR / "session3_status.json").write_text(
        json.dumps(summary, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
    )
    report["artifact_bundle"]["error"] = str(exc)
    DIAGNOSTICS_PATH.write_text(
        json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
    )
print("Copy this complete report and send it for review:", flush=True)
print(json.dumps(report, indent=2, ensure_ascii=False), flush=True)
print(f"Saved copy: {DIAGNOSTICS_PATH}", flush=True)
print(f"Downloadable bundle: {BUNDLE_PATH}", flush=True)
